In [ ]:
from ib_async import *

util.startLoop()

ib = IB()
# Connect asynchronously (safe in notebooks)
if not ib.isConnected():
    await ib.connectAsync("127.0.0.1", 7497, clientId=1)
    print('Connected:', ib.isConnected())



Connected: True


Error 200, reqId 13: No security definition has been found for the request, contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=205.0, right='P', exchange='SMART', tradingClass='QQQ')
Error 200, reqId 14: No security definition has been found for the request, contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=210.0, right='P', exchange='SMART', tradingClass='QQQ')
Error 200, reqId 16: No security definition has been found for the request, contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=220.0, right='P', exchange='SMART', tradingClass='QQQ')
Error 200, reqId 15: No security definition has been found for the request, contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=215.0, right='P', exchange='SMART', tradingClass='QQQ')
Error 200, reqId 17: No security definition has been found for the request, contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=2

In [3]:
index = Index("NDX")
ib.qualifyContracts(index)

[Index(conId=416843, symbol='NDX', exchange='NASDAQ', currency='USD', localSymbol='NDX')]

In [2]:
stock = Stock('QQQ', 'SMART', 'USD')
ib.qualifyContracts(stock)

[Stock(conId=320227571, symbol='QQQ', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='QQQ', tradingClass='NMS')]

In [3]:
ib.reqMarketDataType(4)

In [4]:
[ticker] = ib.reqTickers(stock)
ticker

Ticker(contract=Stock(conId=320227571, symbol='QQQ', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='QQQ', tradingClass='NMS'), time=datetime.datetime(2025, 10, 29, 9, 34, 30, 54407, tzinfo=datetime.timezone.utc), minTick=0.01, bid=634.88, bidSize=400.0, ask=634.91, askSize=100.0, last=634.81, lastSize=100.0, volume=5193.0, close=632.92, halted=0.0, bboExchange='9c0001', snapshotPermissions=3)

In [5]:
spot = ticker.close
spot

632.92

In [6]:
chains = ib.reqSecDefOptParams(stock.symbol, "", stock.secType, stock.conId)

util.df(chains)

,exchange,underlyingConId,tradingClass,multiplier,expirations,strikes
0,AMEX,320227571,QQQ,100,"[20251029, 20251030, 20251031, 20251103, 20251...","[174.78, 179.78, 184.78, 189.78, 194.78, 199.7..."
1,MERCURY,320227571,2QQQ,100,[20251121],"[573.0, 582.0]"
2,NASDAQOM,320227571,2QQQ,100,[20251121],"[573.0, 582.0]"
3,SAPPHIRE,320227571,QQQ,100,"[20251029, 20251030, 20251031, 20251103, 20251...","[174.78, 179.78, 184.78, 189.78, 194.78, 199.7..."
4,ISE,320227571,2QQQ,100,[20251121],"[573.0, 582.0]"
5,NASDAQOM,320227571,QQQ,100,"[20251029, 20251030, 20251031, 20251103, 20251...","[174.78, 179.78, 184.78, 189.78, 194.78, 199.7..."
6,AMEX,320227571,2QQQ,100,[20251121],"[573.0, 582.0]"
7,NASDAQBX,320227571,2QQQ,100,[20251121],"[573.0, 582.0]"
8,MIAX,320227571,QQQ,100,"[20251029, 20251030, 20251031, 20251103, 20251...","[174.78, 179.78, 184.78, 189.78, 194.78, 199.7..."
9,CBOE2,320227571,QQQ,100,"[20251029, 20251030, 20251031, 20251103, 20251...","[174.78, 179.78, 184.78, 189.78, 194.78, 199.7..."


In [7]:
chain = next(c for c in chains if c.tradingClass == stock.symbol and c.exchange == "SMART")
chain

OptionChain(exchange='SMART', underlyingConId='320227571', tradingClass='QQQ', multiplier='100', expirations=['20251029', '20251030', '20251031', '20251103', '20251104', '20251105', '20251106', '20251107', '20251110', '20251111', '20251112', '20251114', '20251121', '20251128', '20251205', '20251219', '20251231', '20260116', '20260130', '20260220', '20260320', '20260331', '20260417', '20260515', '20260618', '20260630', '20260918', '20260930', '20261218', '20270115', '20270617', '20270917', '20271217', '20280121'], strikes=[174.78, 179.78, 184.78, 189.78, 194.78, 199.78, 204.78, 205.0, 209.78, 210.0, 214.78, 215.0, 219.78, 220.0, 224.78, 225.0, 229.78, 230.0, 234.78, 235.0, 239.78, 240.0, 244.78, 245.0, 249.78, 250.0, 254.78, 255.0, 259.78, 260.0, 264.78, 265.0, 269.78, 270.0, 274.78, 275.0, 279.78, 280.0, 284.78, 285.0, 289.78, 290.0, 294.78, 295.0, 299.78, 300.0, 304.78, 305.0, 309.78, 310.0, 314.78, 315.0, 319.78, 320.0, 324.78, 325.0, 329.78, 330.0, 334.78, 335.0, 339.78, 340.0, 344.

In [8]:
strikes = [
    strike
    for strike in chain.strikes
    if strike % 1 == 0 #and spot - 50 < strike < spot + 50
]
expirations = sorted(exp for exp in chain.expirations)[1:3]
rights = ["P", "C"]

contracts = [
    Option(stock.symbol, expiration, strike, right, "SMART", tradingClass=stock.symbol)
    for right in rights
    for expiration in expirations
    for strike in strikes
]

len(contracts)

1304

In [9]:
contracts = ib.qualifyContracts(*contracts)
len(contracts)

Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=205.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=210.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=215.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=220.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=225.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=230.0, right='P', exchange='SMART', tradingClass='QQQ')
Unknown contract: Option(symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=235.0, right='P', exchange='SMART', tradingClass='QQQ')

556

In [10]:

contracts[0]

Option(conId=823916435, symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=500.0, right='P', multiplier='100', exchange='SMART', currency='USD', localSymbol='QQQ   251030P00500000', tradingClass='QQQ')

In [11]:
tickers = ib.reqTickers(*contracts)

tickers[0]

Ticker(contract=Option(conId=823916435, symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=500.0, right='P', multiplier='100', exchange='SMART', currency='USD', localSymbol='QQQ   251030P00500000', tradingClass='QQQ'), time=datetime.datetime(2025, 10, 29, 9, 39, 6, 44182, tzinfo=datetime.timezone.utc), minTick=0.01, bid=-1.0, bidSize=0.0, ask=-1.0, askSize=0.0, close=0.0, bboExchange='c70003', snapshotPermissions=3)

In [14]:
tickers[50:70]


[Ticker(contract=Option(conId=823535261, symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=610.0, right='P', multiplier='100', exchange='SMART', currency='USD', localSymbol='QQQ   251030P00610000', tradingClass='QQQ'), time=datetime.datetime(2025, 10, 29, 9, 39, 10, 892401, tzinfo=datetime.timezone.utc), minTick=0.01, bid=-1.0, bidSize=0.0, ask=-1.0, askSize=0.0, close=0.47, bboExchange='c70003', snapshotPermissions=3),
 Ticker(contract=Option(conId=823918246, symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=611.0, right='P', multiplier='100', exchange='SMART', currency='USD', localSymbol='QQQ   251030P00611000', tradingClass='QQQ'), time=datetime.datetime(2025, 10, 29, 9, 39, 10, 892401, tzinfo=datetime.timezone.utc), minTick=0.01, bid=-1.0, bidSize=0.0, ask=-1.0, askSize=0.0, close=0.52, bboExchange='c70003', snapshotPermissions=3),
 Ticker(contract=Option(conId=823918302, symbol='QQQ', lastTradeDateOrContractMonth='20251030', strike=612.0, right='P', multipl

In [15]:

ib.disconnect()